In [1]:
from open_flamingo import create_model_and_transforms

model, image_processor, tokenizer = create_model_and_transforms(
    clip_vision_encoder_path="ViT-L-14",
    clip_vision_encoder_pretrained="openai",
    lang_encoder_path="anas-awadalla/mpt-1b-redpajama-200b-dolly",
    tokenizer_path="anas-awadalla/mpt-1b-redpajama-200b-dolly",
    cross_attn_every_n_layers=1
)

# grab model checkpoint from huggingface hub
from huggingface_hub import hf_hub_download
import torch

checkpoint_path = hf_hub_download("openflamingo/OpenFlamingo-3B-vitl-mpt1b-langinstruct", "checkpoint.pt")
model.load_state_dict(torch.load(checkpoint_path), strict=False)

/data/2_data_server/cv-07/anaconda3/envs/openflamingo/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/2_data_server/cv-07/anaconda3/envs/openflamingo/lib/python3.9/site-packages/open_clip/factory.py:388: UserWarning: These pretrained weights were trained with QuickGELU activation but the model config does not have that enabled. Consider using a model config with a "-quickgelu" suffix or enable with a flag.
  warnings.warn(
/home/cv-07/.cache/huggingface/modules/transformers_modules/anas-awadalla/mpt-1b-redpajama-200b-dolly/f0a13e41fcee2217cd701219ffa1eaef7fe955ea/attention.py:289: UserWarning: Using `attn_impl: torch`. If your model does not use `alibi` or `prefix_lm` we recommend using `attn_impl: flash` otherwise we recommend using `attn_impl: triton`.
  warnings.warn(


You are using config.init_device='cpu', but you can also use config.init_device="meta" with Composer + FSDP for fast initialization.
Flamingo model initialized with 1046992944 trainable parameters


_IncompatibleKeys(missing_keys=['vision_encoder.class_embedding', 'vision_encoder.positional_embedding', 'vision_encoder.proj', 'vision_encoder.conv1.weight', 'vision_encoder.ln_pre.weight', 'vision_encoder.ln_pre.bias', 'vision_encoder.transformer.resblocks.0.ln_1.weight', 'vision_encoder.transformer.resblocks.0.ln_1.bias', 'vision_encoder.transformer.resblocks.0.attn.in_proj_weight', 'vision_encoder.transformer.resblocks.0.attn.in_proj_bias', 'vision_encoder.transformer.resblocks.0.attn.out_proj.weight', 'vision_encoder.transformer.resblocks.0.attn.out_proj.bias', 'vision_encoder.transformer.resblocks.0.ln_2.weight', 'vision_encoder.transformer.resblocks.0.ln_2.bias', 'vision_encoder.transformer.resblocks.0.mlp.c_fc.weight', 'vision_encoder.transformer.resblocks.0.mlp.c_fc.bias', 'vision_encoder.transformer.resblocks.0.mlp.c_proj.weight', 'vision_encoder.transformer.resblocks.0.mlp.c_proj.bias', 'vision_encoder.transformer.resblocks.1.ln_1.weight', 'vision_encoder.transformer.resbloc

In [10]:
from PIL import Image

# VQA는 한 번에 하나의 이미지만 사용합니다.
query_image = Image.open("/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images_2/TEST_000.jpg").convert("RGB")


# --- 2. 이미지 전처리 (데모 코드 방식 적용) ---
# image_processor를 통해 3D 텐서 [C, H, W] 생성
processed_image = image_processor(query_image)

# .unsqueeze(0)로 배치 차원 추가 -> 4D 텐서 [1, C, H, W]
# 이것이 데모 코드의 vision_x 리스트에 들어가는 각 요소에 해당합니다.
vision_x = processed_image.unsqueeze(0)

# .unsqueeze(1).unsqueeze(0)를 통해 num_media와 num_frames 차원 추가
# 최종적으로 모델이 요구하는 5D 텐서 [1, 1, 1, C, H, W] 생성
# (batch_size=1, num_media=1, num_frames=1)
vision_x = vision_x.unsqueeze(1).unsqueeze(0)
# vision_x = vision_x.to(model.device) # 최종 텐서를 GPU/CPU로 보냄

print(f"모델에 입력될 최종 이미지 텐서의 모양: {vision_x.shape}")


# 이제 inputs 변수는 모델에 입력될 준비가 완벽하게 끝났습니다.
print("최종 입력 텐서의 모양:", inputs.shape)

# OpenFlamingo 형식에 맞는 프롬프트 구성
prompt = "<image>Question: Which of the following foods is not present in the image?\nA. Pizza\nB. Blueberries\nC. Salmon\nD. Avocado\nShort Answer:"

# 토크나이저로 프롬프트를 토큰화
lang_x = tokenizer(
    [prompt],
    return_tensors="pt",
)

모델에 입력될 최종 이미지 텐서의 모양: torch.Size([1, 1, 1, 3, 224, 224])
최종 입력 텐서의 모양: torch.Size([1, 3, 224, 224])


In [11]:
# 질문과 선택지 정의
question = "Which of the following foods is not present in the image?"
options = {
    "A": "Pizza",
    "B": "Blueberries",
    "C": "Salmon",
    "D": "Avocado"
}
choices = list(options.keys()) # ["A", "B", "C", "D"]

# 프롬프트의 기본 구조
# 데모 코드처럼 <image>와 <|endofchunk|>를 사용
prompt_template = "<image>Question: {question}\nA. {A}\nB. {B}\nC. {C}\nD. {D}\nShort Answer: {choice}<|endofchunk|>"

losses = []

print("\n각 선택지에 대한 Loss 계산 중...")
# 각 선택지에 대해 반복
for choice in choices:
    # 각 선택지에 대한 완전한 프롬프트를 구성
    full_prompt = prompt_template.format(
        question=question,
        A=options["A"],
        B=options["B"],
        C=options["C"],
        D=options["D"],
        choice=choice
    )
    
    # 프롬프트를 토큰화
    # padding_side='left'는 generate 시에만 필요하지만, 일관성을 위해 유지 가능
    tokenizer.padding_side = "left"
    lang_x = tokenizer([full_prompt], return_tensors="pt")
    
    # 모델에 입력하여 loss 계산
    with torch.no_grad():
        outputs = model(
            vision_x=vision_x, # 5D 이미지 텐서
            lang_x=lang_x['input_ids'],
            attention_mask=lang_x['attention_mask'],
            labels=lang_x['input_ids'].clone()
        )
        loss = outputs.loss.item()
        losses.append(loss)
        print(f"선택지 '{choice}'의 Loss: {loss:.4f}")

# 가장 loss가 낮은(가장 그럴듯한) 선택지를 정답으로 선택
best_choice_idx = torch.tensor(losses).argmin()
final_answer = choices[best_choice_idx]

print("="*30)
print(f"가장 가능성이 높은 최종 정답: {final_answer}")
print("="*30)


각 선택지에 대한 Loss 계산 중...
선택지 'A'의 Loss: 3.0201
선택지 'B'의 Loss: 2.9710
선택지 'C'의 Loss: 2.9681
선택지 'D'의 Loss: 2.9768
가장 가능성이 높은 최종 정답: C
